In [21]:
!pip -q install requests beautifulsoup4 pandas numpy plotly textblob lxml

In [22]:
import requests
from bs4 import BeautifulSoup
import pandas as pd
import numpy as np
import re
import json
from datetime import datetime
from textblob import TextBlob
from urllib.parse import urljoin

In [23]:
review_sources = {
    "I-8 Markaz": "https://restaurantguru.com/Sugreve-I8-Islamabad",
    "Bahria Phase 7": "https://restaurantguru.com/Sugreve-Blue-Area-Islamabad"
}

for branch, url in review_sources.items():
    print(branch, "→", url)

I-8 Markaz → https://restaurantguru.com/Sugreve-I8-Islamabad
Bahria Phase 7 → https://restaurantguru.com/Sugreve-Blue-Area-Islamabad


In [24]:
headers = {
    "User-Agent": (
        "Mozilla/5.0 (Windows NT 10.0; Win64; x64) "
        "AppleWebKit/537.36 Chrome/153.0 Safari/537.36"
    )
}

pages = {}

for branch, url in review_sources.items():
    response = requests.get(
        url,
        headers=headers,
        timeout=30
    )

    print(branch, response.status_code)

    if response.status_code == 200:
        pages[branch] = response.text
    else:
        print("Could not retrieve:", branch)

I-8 Markaz 200
Bahria Phase 7 200


In [25]:
def extract_reviews(html, branch, source_url):

    soup = BeautifulSoup(html, "html.parser")

    reviews = []

    # Find elements containing Google review text
    possible_blocks = soup.find_all(
        ["div", "article", "section"],
        string=re.compile(r"Google", re.I)
    )

    # Main review containers
    containers = soup.select(
        '[class*="review"], '
        '[class*="Review"], '
        '[itemprop="review"]'
    )

    # Combine possible containers
    all_blocks = containers + possible_blocks

    seen = set()

    for block in all_blocks:

        text = block.get_text(" ", strip=True)

        if len(text) < 30:
            continue

        # Ignore navigation / unrelated text
        if not any(word in text.lower() for word in [
            "food:",
            "service:",
            "atmosphere:",
            "google",
            "review"
        ]):
            continue

        key = text[:300]

        if key in seen:
            continue

        seen.add(key)

        reviews.append({
            "Branch": branch,
            "Raw_Text": text,
            "Source_URL": source_url
        })

    return reviews


raw_reviews = []

for branch, html in pages.items():

    extracted = extract_reviews(
        html,
        branch,
        review_sources[branch]
    )

    raw_reviews.extend(extracted)

print("Raw review blocks found:", len(raw_reviews))

Raw review blocks found: 11


In [26]:
reviews_df = pd.DataFrame(raw_reviews)

if reviews_df.empty:
    raise ValueError(
        "No reviews were extracted. The public page may have changed "
        "its HTML structure or blocked automated access."
    )

reviews_df = reviews_df.drop_duplicates(
    subset=["Branch", "Raw_Text"]
).reset_index(drop=True)

print("Unique public review blocks:", len(reviews_df))

reviews_df.head()

Unique public review blocks: 11


,Branch,Raw_Text,Source_URL
0,I-8 Markaz,Visitors' reviews on Sugreve - I-8 Markaz / 136,https://restaurantguru.com/Sugreve-I8-Islamabad
1,I-8 Markaz,Request content removal arooj ali 2 months ago...,https://restaurantguru.com/Sugreve-I8-Islamabad
2,I-8 Markaz,Request content removal arooj ali 2 months ago...,https://restaurantguru.com/Sugreve-I8-Islamabad
3,I-8 Markaz,Request content removal Mughal Tools 2 months ...,https://restaurantguru.com/Sugreve-I8-Islamabad
4,I-8 Markaz,S Response from the owner 2 months ago Thank y...,https://restaurantguru.com/Sugreve-I8-Islamabad


In [27]:
def extract_rating(text, label):

    pattern = rf"{label}\s*:\s*(\d(?:\.\d)?)"

    match = re.search(
        pattern,
        text,
        flags=re.IGNORECASE
    )

    if match:
        return float(match.group(1))

    return np.nan


reviews_df["Food_Rating"] = reviews_df["Raw_Text"].apply(
    lambda x: extract_rating(x, "Food")
)

reviews_df["Service_Rating"] = reviews_df["Raw_Text"].apply(
    lambda x: extract_rating(x, "Service")
)

reviews_df["Atmosphere_Rating"] = reviews_df["Raw_Text"].apply(
    lambda x: extract_rating(x, "Atmosphere")
)

reviews_df["Overall_Rating"] = reviews_df[
    ["Food_Rating", "Service_Rating", "Atmosphere_Rating"]
].mean(axis=1)

reviews_df.head()

,Branch,Raw_Text,Source_URL,Food_Rating,Service_Rating,Atmosphere_Rating,Overall_Rating
0,I-8 Markaz,Visitors' reviews on Sugreve - I-8 Markaz / 136,https://restaurantguru.com/Sugreve-I8-Islamabad,NaN,NaN,NaN,NaN
1,I-8 Markaz,Request content removal arooj ali 2 months ago...,https://restaurantguru.com/Sugreve-I8-Islamabad,5.0,5.0,5.0,5.0
2,I-8 Markaz,Request content removal arooj ali 2 months ago...,https://restaurantguru.com/Sugreve-I8-Islamabad,5.0,5.0,5.0,5.0
3,I-8 Markaz,Request content removal Mughal Tools 2 months ...,https://restaurantguru.com/Sugreve-I8-Islamabad,5.0,5.0,5.0,5.0
4,I-8 Markaz,S Response from the owner 2 months ago Thank y...,https://restaurantguru.com/Sugreve-I8-Islamabad,NaN,NaN,NaN,NaN


In [28]:
def extract_field(text, field):

    pattern = rf"{field}\s*:\s*([^|]+?)(?=\s+(?:Meal type|Price per person|Food|Service|Atmosphere):|$)"

    match = re.search(
        pattern,
        text,
        flags=re.IGNORECASE
    )

    if match:
        return match.group(1).strip()

    return np.nan


reviews_df["Order_Type"] = reviews_df["Raw_Text"].apply(
    lambda x: extract_field(x, "Order type")
)

reviews_df["Meal_Type"] = reviews_df["Raw_Text"].apply(
    lambda x: extract_field(x, "Meal type")
)

reviews_df["Price_Per_Person"] = reviews_df["Raw_Text"].apply(
    lambda x: extract_field(x, "Price per person")
)

reviews_df[
    [
        "Branch",
        "Food_Rating",
        "Service_Rating",
        "Atmosphere_Rating",
        "Order_Type",
        "Meal_Type",
        "Price_Per_Person"
    ]
].head()

,Branch,Food_Rating,Service_Rating,Atmosphere_Rating,Order_Type,Meal_Type,Price_Per_Person
0,I-8 Markaz,NaN,NaN,NaN,NaN,NaN,NaN
1,I-8 Markaz,5.0,5.0,5.0,Delivery,NaN,"Rs 3,000–4,000 S Response from the owner 2 mon..."
2,I-8 Markaz,5.0,5.0,5.0,NaN,NaN,NaN
3,I-8 Markaz,5.0,5.0,5.0,Delivery,NaN,"Rs 3,000–4,000 S Response from the owner 2 mon..."
4,I-8 Markaz,NaN,NaN,NaN,NaN,NaN,NaN


In [29]:
def extract_reviewer(text):

    # Common pattern used on public review pages
    match = re.search(
        r"(?:Image:\s*)?([A-Z][A-Za-z0-9_.-]*(?:\s+[A-Z][A-Za-z0-9_.-]*){0,3})\s+(?:a month|2 months|3 months|4 months|5 months|6 months|a year|years?)",
        text
    )

    if match:
        return match.group(1).strip()

    return "Public reviewer"


reviews_df["Reviewer"] = reviews_df["Raw_Text"].apply(
    extract_reviewer
)

In [30]:
def clean_review_text(text):

    text = re.sub(
        r"Food\s*:\s*\d(?:\.\d)?",
        "",
        text,
        flags=re.I
    )

    text = re.sub(
        r"Service\s*:\s*\d(?:\.\d)?",
        "",
        text,
        flags=re.I
    )

    text = re.sub(
        r"Atmosphere\s*:\s*\d(?:\.\d)?",
        "",
        text,
        flags=re.I
    )

    text = re.sub(
        r"Order type\s*:.*?(?=Meal type|Price per person|$)",
        "",
        text,
        flags=re.I
    )

    text = re.sub(
        r"Meal type\s*:.*?(?=Price per person|$)",
        "",
        text,
        flags=re.I
    )

    text = re.sub(
        r"Price per person\s*:.*?$",
        "",
        text,
        flags=re.I
    )

    return text.strip()


reviews_df["Review_Text"] = reviews_df["Raw_Text"].apply(
    clean_review_text
)

reviews_df["Review_Excerpt"] = (
    reviews_df["Review_Text"]
    .str.replace(r"\s+", " ", regex=True)
    .str.slice(0, 280)
)

reviews_df[
    ["Reviewer", "Branch", "Review_Excerpt"]
].head()

,Reviewer,Branch,Review_Excerpt
0,Public reviewer,I-8 Markaz,Visitors' reviews on Sugreve - I-8 Markaz / 136
1,Mughal Tools,I-8 Markaz,Request content removal arooj ali 2 months ago...
2,Public reviewer,I-8 Markaz,Request content removal arooj ali 2 months ago...
3,Mughal Tools,I-8 Markaz,Request content removal Mughal Tools 2 months ...
4,Public reviewer,I-8 Markaz,S Response from the owner 2 months ago Thank y...


In [31]:
def sentiment_score(text):

    if not isinstance(text, str) or not text.strip():
        return 0

    return TextBlob(text).sentiment.polarity


def sentiment_label(score):

    if score > 0.10:
        return "Positive"

    elif score < -0.10:
        return "Negative"

    else:
        return "Neutral"


reviews_df["Sentiment_Score"] = reviews_df[
    "Review_Text"
].apply(sentiment_score)

reviews_df["Sentiment"] = reviews_df[
    "Sentiment_Score"
].apply(sentiment_label)

reviews_df[
    [
        "Reviewer",
        "Sentiment",
        "Sentiment_Score"
    ]
].head()

,Reviewer,Sentiment,Sentiment_Score
0,Public reviewer,Neutral,0.000000
1,Mughal Tools,Positive,0.284091
2,Public reviewer,Positive,0.284091
3,Mughal Tools,Neutral,0.000000
4,Public reviewer,Neutral,0.000000


In [32]:
products = [
    "Chocolate Malt Cake",
    "Mango Classic Cake",
    "Three Milk Cake",
    "Lotus",
    "Lotus Milk Cake",
    "Banana Pudding",
    "Strawberry Cheesecake",
    "Cheesecake",
    "Chocolate Cake",
    "Chocolate",
    "Cookies",
    "Sundae",
    "Ice Cream",
    "Coffee",
    "Matilda Cake"
]


def find_products(text):

    if not isinstance(text, str):
        return "Not specified"

    found = []

    text_lower = text.lower()

    for product in products:

        if product.lower() in text_lower:
            found.append(product)

    if found:
        return ", ".join(sorted(set(found)))

    return "Not specified"


reviews_df["Products_Mentioned"] = reviews_df[
    "Review_Text"
].apply(find_products)

In [33]:
def extract_date(text):

    patterns = [
        r"(\d{1,2}\s+\w+\s+\d{4})",
        r"(\w+\s+\d{1,2},\s+\d{4})"
    ]

    for pattern in patterns:

        match = re.search(pattern, text)

        if match:
            return match.group(1)

    return np.nan


reviews_df["Review_Date"] = reviews_df[
    "Raw_Text"
].apply(extract_date)

In [34]:
final_reviews = reviews_df[
    [
        "Reviewer",
        "Branch",
        "Review_Date",
        "Overall_Rating",
        "Food_Rating",
        "Service_Rating",
        "Atmosphere_Rating",
        "Products_Mentioned",
        "Order_Type",
        "Meal_Type",
        "Price_Per_Person",
        "Sentiment",
        "Sentiment_Score",
        "Review_Excerpt",
        "Source_URL"
    ]
].copy()

final_reviews = final_reviews.drop_duplicates()

print("FINAL PUBLIC REVIEW RECORDS:", len(final_reviews))

final_reviews.head()

FINAL PUBLIC REVIEW RECORDS: 11


,Reviewer,Branch,Review_Date,Overall_Rating,Food_Rating,Service_Rating,Atmosphere_Rating,Products_Mentioned,Order_Type,Meal_Type,Price_Per_Person,Sentiment,Sentiment_Score,Review_Excerpt,Source_URL
0,Public reviewer,I-8 Markaz,NaN,NaN,NaN,NaN,NaN,Not specified,NaN,NaN,NaN,Neutral,0.000000,Visitors' reviews on Sugreve - I-8 Markaz / 136,https://restaurantguru.com/Sugreve-I8-Islamabad
1,Mughal Tools,I-8 Markaz,NaN,5.0,5.0,5.0,5.0,Not specified,Delivery,NaN,"Rs 3,000–4,000 S Response from the owner 2 mon...",Positive,0.284091,Request content removal arooj ali 2 months ago...,https://restaurantguru.com/Sugreve-I8-Islamabad
2,Public reviewer,I-8 Markaz,NaN,5.0,5.0,5.0,5.0,Not specified,NaN,NaN,NaN,Positive,0.284091,Request content removal arooj ali 2 months ago...,https://restaurantguru.com/Sugreve-I8-Islamabad
3,Mughal Tools,I-8 Markaz,NaN,5.0,5.0,5.0,5.0,Not specified,Delivery,NaN,"Rs 3,000–4,000 S Response from the owner 2 mon...",Neutral,0.000000,Request content removal Mughal Tools 2 months ...,https://restaurantguru.com/Sugreve-I8-Islamabad
4,Public reviewer,I-8 Markaz,NaN,NaN,NaN,NaN,NaN,Not specified,NaN,NaN,NaN,Neutral,0.000000,S Response from the owner 2 months ago Thank y...,https://restaurantguru.com/Sugreve-I8-Islamabad


In [35]:
print(
    final_reviews["Branch"].value_counts()
)

print(
    final_reviews["Sentiment"].value_counts()
)

Branch
I-8 Markaz        7
Bahria Phase 7    4
Name: count, dtype: int64
Sentiment
Neutral     8
Positive    3
Name: count, dtype: int64


In [36]:
final_reviews.to_csv(
    "Sugreve_Public_Reviews.csv",
    index=False
)

print("Saved: Sugreve_Public_Reviews.csv")

Saved: Sugreve_Public_Reviews.csv


In [37]:
products_df = pd.DataFrame({

    "Product": [
        "Lotus Blondie",
        "Rafaello Dream Cake",
        "Mini Rafaello Dream Cake",
        "Three Milk Cake",
        "Nutella Sundae",
        "New York Cheesecake",
        "Double Chocolate Cookie",
        "Cherry Chocolate Dream Cake",
        "Lotus Cream Cheesecake",
        "Matilda Cake Tub",
        "Nutella Brownie",
        "Nutella Cake",
        "Classic Praline Cake",
        "Malt Icing Brownie",
        "Chocolate Fudge Icing Brownie",
        "Blueberry Cheesecake"
    ],

    "Category": [
        "Blondie",
        "Cake",
        "Cake",
        "Cake",
        "Sundae",
        "Cheesecake",
        "Cookie",
        "Cake",
        "Cheesecake",
        "Cake Tub",
        "Brownie",
        "Cake",
        "Cake",
        "Brownie",
        "Brownie",
        "Cheesecake"
    ]
})

products_df

,Product,Category
0,Lotus Blondie,Blondie
1,Rafaello Dream Cake,Cake
2,Mini Rafaello Dream Cake,Cake
3,Three Milk Cake,Cake
4,Nutella Sundae,Sundae
5,New York Cheesecake,Cheesecake
6,Double Chocolate Cookie,Cookie
7,Cherry Chocolate Dream Cake,Cake
8,Lotus Cream Cheesecake,Cheesecake
9,Matilda Cake Tub,Cake Tub


In [38]:
import json
from datetime import datetime

# -----------------------------------------
# Prepare data
# -----------------------------------------

dashboard_df = final_reviews.copy()

dashboard_df = dashboard_df.fillna("Not specified")

data_json = dashboard_df.to_json(
    orient="records",
    force_ascii=False
)

generated_date = datetime.now().strftime("%d %b %Y")


# -----------------------------------------
# HTML
# -----------------------------------------

html = r"""
<!DOCTYPE html>

<html>

<head>

<meta charset="UTF-8">

<meta name="viewport"
      content="width=device-width, initial-scale=1.0">

<title>Sugreve Customer & Product Intelligence Dashboard</title>

<script src="https://cdn.plot.ly/plotly-2.35.2.min.js"></script>

<style>

* {
    box-sizing: border-box;
}

body {
    margin: 0;
    font-family: Arial, Helvetica, sans-serif;
    background: #F8F4FB;
    color: #281B31;
}

.header {
    background:
        linear-gradient(135deg, #4B1F6F, #7B3FA1);

    color: white;

    padding: 32px 40px;

    border-radius: 0 0 28px 28px;
}

.header h1 {
    margin: 0;
    font-size: 30px;
}

.header p {
    margin: 8px 0 0;
    opacity: 0.85;
}

.badge {
    display: inline-block;
    margin-top: 16px;
    padding: 7px 13px;
    border-radius: 20px;
    background: rgba(255,255,255,0.15);
    font-size: 12px;
}

.container {
    max-width: 1500px;
    margin: auto;
    padding: 28px;
}

.filters {
    background: white;
    padding: 22px;
    border-radius: 20px;
    box-shadow: 0 8px 25px rgba(91,42,134,0.08);
    margin-bottom: 22px;
}

.filters-grid {
    display: grid;
    grid-template-columns:
        repeat(auto-fit, minmax(190px, 1fr));

    gap: 15px;
}

label {
    display: block;
    font-size: 12px;
    font-weight: bold;
    color: #68457B;
    margin-bottom: 7px;
}

select,
input {
    width: 100%;
    padding: 11px;
    border: 1px solid #DCC7E8;
    border-radius: 10px;
    background: #FCFAFD;
    color: #281B31;
}

button {
    border: none;
    border-radius: 10px;
    padding: 11px 18px;
    cursor: pointer;
    font-weight: bold;
}

.reset {
    background: #5B2A86;
    color: white;
}

.export {
    background: #EADCF2;
    color: #4B1F6F;
}

.kpis {
    display: grid;
    grid-template-columns:
        repeat(auto-fit, minmax(210px, 1fr));

    gap: 18px;

    margin-bottom: 22px;
}

.kpi {
    background: white;
    border-radius: 18px;
    padding: 22px;

    box-shadow:
        0 8px 25px rgba(91,42,134,0.08);

    border-top: 4px solid #7B3FA1;
}

.kpi-title {
    color: #765A84;
    font-size: 13px;
}

.kpi-value {
    font-size: 29px;
    font-weight: bold;
    margin-top: 8px;
    color: #4B1F6F;
}

.charts {
    display: grid;
    grid-template-columns:
        repeat(auto-fit, minmax(440px, 1fr));

    gap: 20px;
}

.chart-card {
    background: white;
    border-radius: 20px;
    padding: 15px;

    box-shadow:
        0 8px 25px rgba(91,42,134,0.07);
}

.chart {
    width: 100%;
    height: 390px;
}

.section {
    background: white;
    margin-top: 22px;
    padding: 22px;
    border-radius: 20px;
    box-shadow: 0 8px 25px rgba(91,42,134,0.07);
}

.section h2 {
    color: #4B1F6F;
}

table {
    width: 100%;
    border-collapse: collapse;
}

th {
    background: #5B2A86;
    color: white;
    padding: 12px;
    text-align: left;
}

td {
    padding: 11px;
    border-bottom: 1px solid #EEE4F3;
    font-size: 13px;
}

tr:hover {
    background: #F8F0FC;
}

.sentiment {
    padding: 5px 9px;
    border-radius: 20px;
    font-size: 11px;
    font-weight: bold;
}

.source {
    color: #6D4084;
    text-decoration: none;
}

.insights {
    display: grid;
    grid-template-columns:
        repeat(auto-fit, minmax(230px, 1fr));

    gap: 15px;
}

.insight {
    background: #F7F0FA;
    padding: 18px;
    border-radius: 14px;
}

.footer {
    text-align: center;
    padding: 30px;
    color: #80688D;
    font-size: 12px;
}

@media(max-width:700px) {

    .header {
        padding: 25px;
    }

    .container {
        padding: 15px;
    }

    .charts {
        grid-template-columns: 1fr;
    }

}

</style>

</head>


<body>


<div class="header">

    <h1>
        Sugreve Customer & Product Intelligence
    </h1>

    <p>
        Public Review Analytics • Product Intelligence • Branch Insights
    </p>

    <span class="badge">
        PUBLIC DATA — NOT INTERNAL SUGREVE DATA
    </span>

</div>


<div class="container">


<!-- FILTERS -->

<div class="filters">

    <div class="filters-grid">

        <div>

            <label>BRANCH</label>

            <select id="branchFilter">
                <option value="">All Branches</option>
            </select>

        </div>


        <div>

            <label>RATING</label>

            <select id="ratingFilter">

                <option value="">All Ratings</option>
                <option value="5">5 Star</option>
                <option value="4">4 Star</option>
                <option value="3">3 Star</option>
                <option value="2">2 Star</option>
                <option value="1">1 Star</option>

            </select>

        </div>


        <div>

            <label>SENTIMENT</label>

            <select id="sentimentFilter">

                <option value="">All Sentiment</option>
                <option value="Positive">Positive</option>
                <option value="Neutral">Neutral</option>
                <option value="Negative">Negative</option>

            </select>

        </div>


        <div>

            <label>PRODUCT</label>

            <select id="productFilter">

                <option value="">All Products</option>

            </select>

        </div>


        <div>

            <label>SEARCH REVIEWS</label>

            <input
                id="searchFilter"
                type="text"
                placeholder="Search chocolate, service, cake..."
            >

        </div>


        <div>

            <label>ACTIONS</label>

            <button
                class="reset"
                onclick="resetFilters()"
            >
                Reset Filters
            </button>

            <button
                class="export"
                onclick="downloadCSV()"
            >
                Export CSV
            </button>

        </div>

    </div>

</div>


<!-- KPIs -->

<div class="kpis">

    <div class="kpi">

        <div class="kpi-title">
            PUBLIC REVIEWS
        </div>

        <div
            class="kpi-value"
            id="kpiReviews"
        >
            0
        </div>

    </div>


    <div class="kpi">

        <div class="kpi-title">
            AVERAGE RATING
        </div>

        <div
            class="kpi-value"
            id="kpiRating"
        >
            —
        </div>

    </div>


    <div class="kpi">

        <div class="kpi-title">
            BRANCHES
        </div>

        <div
            class="kpi-value"
            id="kpiBranches"
        >
            0
        </div>

    </div>


    <div class="kpi">

        <div class="kpi-title">
            POSITIVE REVIEWS
        </div>

        <div
            class="kpi-value"
            id="kpiPositive"
        >
            0%
        </div>

    </div>

</div>


<!-- CHARTS -->

<div class="charts">


    <div class="chart-card">

        <div
            id="ratingChart"
            class="chart"
        ></div>

    </div>


    <div class="chart-card">

        <div
            id="sentimentChart"
            class="chart"
        ></div>

    </div>


    <div class="chart-card">

        <div
            id="branchChart"
            class="chart"
        ></div>

    </div>


    <div class="chart-card">

        <div
            id="productChart"
            class="chart"
        ></div>

    </div>


</div>


<!-- INSIGHTS -->

<div class="section">

    <h2>
        Live Business Insights
    </h2>

    <div
        class="insights"
        id="insights"
    ></div>

</div>


<!-- REVIEW TABLE -->

<div class="section">

    <h2>
        Customer Review Explorer
    </h2>

    <div style="overflow-x:auto">

        <table>

            <thead>

                <tr>

                    <th>Reviewer</th>
                    <th>Branch</th>
                    <th>Rating</th>
                    <th>Product</th>
                    <th>Sentiment</th>
                    <th>Review</th>
                    <th>Source</th>

                </tr>

            </thead>

            <tbody id="reviewTable"></tbody>

        </table>

    </div>

</div>


<div class="footer">

    Generated __GENERATED_DATE__ •
    Publicly available review analysis •
    Review source attribution preserved

</div>


</div>


<script>

const DATA = __DATA_JSON__;


const purple = "#5B2A86";
const purple2 = "#7B3FA1";
const lightPurple = "#DCC7E8";


function uniqueValues(field) {

    return [...new Set(
        DATA
        .map(x => x[field])
        .filter(x => x && x !== "Not specified")
    )].sort();

}


function populateFilters() {

    const branches = uniqueValues("Branch");

    const products = uniqueValues("Products_Mentioned");


    const branchSelect =
        document.getElementById("branchFilter");

    branches.forEach(x => {

        const option =
            document.createElement("option");

        option.value = x;
        option.textContent = x;

        branchSelect.appendChild(option);

    });


    const productSelect =
        document.getElementById("productFilter");

    products.forEach(x => {

        const option =
            document.createElement("option");

        option.value = x;
        option.textContent = x;

        productSelect.appendChild(option);

    });

}


function getFilteredData() {

    const branch =
        document.getElementById("branchFilter").value;

    const rating =
        document.getElementById("ratingFilter").value;

    const sentiment =
        document.getElementById("sentimentFilter").value;

    const product =
        document.getElementById("productFilter").value;

    const search =
        document
        .getElementById("searchFilter")
        .value
        .toLowerCase();


    return DATA.filter(row => {

        const ratingValue =
            parseFloat(row.Overall_Rating);


        const text =
            String(row.Review_Excerpt || "")
            .toLowerCase();


        const productText =
            String(row.Products_Mentioned || "")
            .toLowerCase();


        return (

            (!branch ||
             row.Branch === branch)

            &&

            (!rating ||
             Math.round(ratingValue) ===
             Number(rating))

            &&

            (!sentiment ||
             row.Sentiment === sentiment)

            &&

            (!product ||
             row.Products_Mentioned.includes(product))

            &&

            (!search ||
             text.includes(search) ||
             productText.includes(search))

        );

    });

}


function updateKPIs(data) {

    document.getElementById("kpiReviews")
        .textContent = data.length;


    const ratings =
        data
        .map(x => Number(x.Overall_Rating))
        .filter(x => !isNaN(x) && x > 0);


    const avg =
        ratings.length
        ? ratings.reduce((a,b) => a+b,0) / ratings.length
        : 0;


    document.getElementById("kpiRating")
        .textContent =
        ratings.length
        ? avg.toFixed(2) + " / 5"
        : "—";


    const branches =
        new Set(data.map(x => x.Branch));


    document.getElementById("kpiBranches")
        .textContent = branches.size;


    const positive =
        data.filter(
            x => x.Sentiment === "Positive"
        ).length;


    const positivePct =
        data.length
        ? Math.round(positive / data.length * 100)
        : 0;


    document.getElementById("kpiPositive")
        .textContent = positivePct + "%";

}


function baseLayout(title) {

    return {

        title: {
            text: title,
            font: {
                size: 17,
                color: "#4B1F6F"
            }
        },

        paper_bgcolor: "white",

        plot_bgcolor: "white",

        margin: {
            l: 55,
            r: 20,
            t: 55,
            b: 55
        },

        font: {
            family: "Arial",
            color: "#4B1F6F"
        },

        hoverlabel: {
            bgcolor: "white",
            bordercolor: purple
        }

    };

}


function updateCharts(data) {


    // -----------------------------
    // Rating chart
    // -----------------------------

    const ratings = [1,2,3,4,5];

    const ratingCounts =
        ratings.map(
            r =>
            data.filter(
                x =>
                Math.round(
                    Number(x.Overall_Rating)
                ) === r
            ).length
        );


    Plotly.react(
        "ratingChart",

        [{
            x: ratings,
            y: ratingCounts,
            type: "bar",
            marker: {
                color: purple2
            }
        }],

        baseLayout("Rating Distribution"),

        {
            responsive: true,
            displayModeBar: false
        }

    );


    // -----------------------------
    // Sentiment chart
    // -----------------------------

    const sentiments =
        ["Positive","Neutral","Negative"];


    const sentimentCounts =
        sentiments.map(
            s =>
            data.filter(
                x => x.Sentiment === s
            ).length
        );


    Plotly.react(
        "sentimentChart",

        [{
            labels: sentiments,
            values: sentimentCounts,
            type: "pie",
            hole: 0.58,
            marker: {
                colors: [
                    "#7B3FA1",
                    "#C9B0D9",
                    "#4B1F6F"
                ]
            }
        }],

        baseLayout("Customer Sentiment"),

        {
            responsive: true,
            displayModeBar: false
        }

    );


    // -----------------------------
    // Branch chart
    // -----------------------------

    const branchMap = {};

    data.forEach(x => {

        branchMap[x.Branch] =
            (branchMap[x.Branch] || 0) + 1;

    });


    const branchNames =
        Object.keys(branchMap);

    const branchValues =
        Object.values(branchMap);


    Plotly.react(
        "branchChart",

        [{
            x: branchValues,
            y: branchNames,
            type: "bar",
            orientation: "h",
            marker: {
                color: purple
            }
        }],

        baseLayout("Reviews by Branch"),

        {
            responsive: true,
            displayModeBar: false
        }

    );


    // -----------------------------
    // Product chart
    // -----------------------------

    const productMap = {};


    data.forEach(x => {

        const items =
            String(x.Products_Mentioned)
            .split(",");


        items.forEach(product => {

            product = product.trim();

            if (
                product &&
                product !== "Not specified"
            ) {

                productMap[product] =
                    (productMap[product] || 0) + 1;

            }

        });

    });


    const topProducts =
        Object.entries(productMap)
        .sort((a,b) => b[1] - a[1])
        .slice(0,10);


    Plotly.react(
        "productChart",

        [{
            x: topProducts.map(x => x[1]),
            y: topProducts.map(x => x[0]),
            type: "bar",
            orientation: "h",
            marker: {
                color: purple2
            }
        }],

        baseLayout("Most Mentioned Products"),

        {
            responsive: true,
            displayModeBar: false
        }

    );

}


function updateInsights(data) {

    const box =
        document.getElementById("insights");


    if (!data.length) {

        box.innerHTML =
            "<div class='insight'>No records match the selected filters.</div>";

        return;

    }


    const ratings =
        data
        .map(x => Number(x.Overall_Rating))
        .filter(x => !isNaN(x));


    const average =
        ratings.reduce((a,b) => a+b,0)
        / ratings.length;


    const positive =
        data.filter(
            x => x.Sentiment === "Positive"
        ).length;


    const topBranch =
        Object.entries(
            data.reduce((acc,x) => {

                acc[x.Branch] =
                    (acc[x.Branch] || 0) + 1;

                return acc;

            }, {})
        )
        .sort((a,b) => b[1]-a[1])[0];


    const topProduct =
        Object.entries(
            data.reduce((acc,x) => {

                String(x.Products_Mentioned)
                .split(",")
                .forEach(p => {

                    p = p.trim();

                    if (
                        p &&
                        p !== "Not specified"
                    ) {

                        acc[p] =
                            (acc[p] || 0) + 1;

                    }

                });

                return acc;

            }, {})
        )
        .sort((a,b) => b[1]-a[1])[0];


    box.innerHTML = `

        <div class="insight">

            <b>Average Experience</b>

            <br><br>

            ${average.toFixed(2)} / 5

        </div>


        <div class="insight">

            <b>Positive Sentiment</b>

            <br><br>

            ${Math.round(
                positive / data.length * 100
            )}%

        </div>


        <div class="insight">

            <b>Most Reviewed Branch</b>

            <br><br>

            ${topBranch ? topBranch[0] : "—"}

        </div>


        <div class="insight">

            <b>Most Mentioned Product</b>

            <br><br>

            ${topProduct ? topProduct[0] : "—"}

        </div>

    `;

}


function updateTable(data) {

    const table =
        document.getElementById(
            "reviewTable"
        );


    const rows =
        data.slice(0,100);


    table.innerHTML =
        rows.map(row => {

            const rating =
                Number(row.Overall_Rating);


            return `

            <tr>

                <td>
                    ${escapeHTML(row.Reviewer)}
                </td>

                <td>
                    ${escapeHTML(row.Branch)}
                </td>

                <td>
                    ${isNaN(rating)
                    ? "—"
                    : rating.toFixed(1)}
                </td>

                <td>
                    ${escapeHTML(
                        row.Products_Mentioned
                    )}
                </td>

                <td>

                    <span class="sentiment">

                        ${escapeHTML(
                            row.Sentiment
                        )}

                    </span>

                </td>

                <td>
                    ${escapeHTML(
                        row.Review_Excerpt
                    )}
                </td>

                <td>

                    <a
                        class="source"
                        href="${row.Source_URL}"
                        target="_blank"
                    >
                        View Source
                    </a>

                </td>

            </tr>

            `;

        }).join("");

}


function escapeHTML(value) {

    return String(value ?? "")
        .replace(/&/g, "&amp;")
        .replace(/</g, "&lt;")
        .replace(/>/g, "&gt;")
        .replace(/"/g, "&quot;")
        .replace(/'/g, "&#039;");

}


function refreshDashboard() {

    const filtered =
        getFilteredData();


    updateKPIs(filtered);

    updateCharts(filtered);

    updateInsights(filtered);

    updateTable(filtered);

}


function resetFilters() {

    document.getElementById(
        "branchFilter"
    ).value = "";

    document.getElementById(
        "ratingFilter"
    ).value = "";

    document.getElementById(
        "sentimentFilter"
    ).value = "";

    document.getElementById(
        "productFilter"
    ).value = "";

    document.getElementById(
        "searchFilter"
    ).value = "";

    refreshDashboard();

}


function downloadCSV() {

    const data =
        getFilteredData();


    if (!data.length) {

        alert("No filtered records to export.");

        return;

    }


    const headers =
        Object.keys(data[0]);


    const csv = [

        headers.join(","),

        ...data.map(row =>
            headers.map(
                h =>
                `"${String(
                    row[h] ?? ""
                ).replace(/"/g,'""')}"`
            ).join(",")
        )

    ].join("\n");


    const blob =
        new Blob(
            [csv],
            {type: "text/csv;charset=utf-8;"}
        );


    const url =
        URL.createObjectURL(blob);


    const a =
        document.createElement("a");


    a.href = url;

    a.download =
        "Sugreve_Filtered_Public_Reviews.csv";

    a.click();


    URL.revokeObjectURL(url);

}


document
.getElementById("branchFilter")
.addEventListener(
    "change",
    refreshDashboard
);


document
.getElementById("ratingFilter")
.addEventListener(
    "change",
    refreshDashboard
);


document
.getElementById("sentimentFilter")
.addEventListener(
    "change",
    refreshDashboard
);


document
.getElementById("productFilter")
.addEventListener(
    "change",
    refreshDashboard
);


document
.getElementById("searchFilter")
.addEventListener(
    "input",
    refreshDashboard
);


populateFilters();

refreshDashboard();

</script>


</body>

</html>
"""


html = (
    html
    .replace("__DATA_JSON__", data_json)
    .replace(
        "__GENERATED_DATE__",
        generated_date
    )
)


with open(
    "Sugreve_Customer_Product_Intelligence_Dashboard.html",
    "w",
    encoding="utf-8"
) as f:

    f.write(html)


print(
    "Dashboard created successfully!"
)

print(
    "File: Sugreve_Customer_Product_Intelligence_Dashboard.html"
)

Dashboard created successfully!
File: Sugreve_Customer_Product_Intelligence_Dashboard.html


In [39]:
from google.colab import files

files.download("Sugreve_Customer_Product_Intelligence_Dashboard.html")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>